In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================
import os, random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm
from tqdm import tqdm
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Imports done | CUDA:', torch.cuda.is_available())

Imports done | CUDA: True


In [20]:
BASE = '/kaggle/input/datasets/madhvii0911/uercccc/UERC26_Oriented/data/sequestered_anonymized'
# Agar file not found aa raha hai, toh check karo ki image '0' folder ke andar hai ya nahi
IMG_DIR = BASE 
CSV_PATH = os.path.join(BASE, 'image_list.csv')

df = pd.read_csv(CSV_PATH, names=['img_path'])
df['person_id'] = [i // 10 for i in range(len(df))]

all_pids = df['person_id'].unique()
np.random.shuffle(all_pids)
split = int(0.8 * len(all_pids))

train_df = df[df['person_id'].isin(all_pids[:split])].reset_index(drop=True)
val_df = df[df['person_id'].isin(all_pids[split:])].reset_index(drop=True)

In [37]:
import os

# Check folder '0' directly
folder_0 = os.path.join(BASE, '0')
if os.path.exists(folder_0):
    files = os.listdir(folder_0)
    print(f"✅ Folder '0' mil gaya! Isme {len(files)} files hain.")
    if len(files) > 0:
        sample_file = files[0]
        print(f"Sample file name: '{sample_file}'") # Iska extension dhyan se dekho
        
        # Ab check karo full path work kar raha hai ya nahi
        full_path_test = os.path.join(folder_0, sample_file)
        print(f"Full path test: {full_path_test}")
        print(f"Does it exist? {os.path.exists(full_path_test)}")
else:
    print("❌ Folder '0' nahi mila. BASE path check karo.")

✅ Folder '0' mil gaya! Isme 1670 files hain.
Sample file name: '1126.webp'
Full path test: /kaggle/input/datasets/madhvii0911/uercccc/UERC26_Oriented/data/sequestered_anonymized/0/1126.webp
Does it exist? True


In [38]:
class SiameseDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        # IDs grouping
        self.id2imgs = self.df.groupby('person_id')['img_path'].apply(list).to_dict()
        self.all_pids = list(self.id2imgs.keys())

    def __len__(self):
        return len(self.df)

    def load_img(self, path):
        # 1. Path ko string banao aur clean karo
        path = str(path).strip()
        
        # 2. Agar path mein extension galat hai ya nahi hai, toh fix karo
        # Example: '0/1126.png' -> '0/1126.webp'
        if not path.endswith('.webp'):
            path = os.path.splitext(path)[0] + '.webp'
        
        full_path = os.path.join(self.img_dir, path)
        
        if os.path.exists(full_path):
            return Image.open(full_path).convert('RGB')
        else:
            # Agar file nahi milti toh error throw karein taaki retry loop chale
            raise FileNotFoundError(f"Missing: {full_path}")

    def __getitem__(self, idx):
        for _ in range(20): # Retry loop
            try:
                row = self.df.iloc[idx]
                target = random.randint(0, 1)
                img1 = self.load_img(row['img_path'])
                
                if target == 1:
                    img2_path = random.choice(self.id2imgs[row['person_id']])
                else:
                    other_pid = random.choice([p for p in self.all_pids if p != row['person_id']])
                    img2_path = random.choice(self.id2imgs[other_pid])
                
                img2 = self.load_img(img2_path)
                
                if self.transform:
                    img1, img2 = self.transform(img1), self.transform(img2)
                return img1, img2, torch.tensor(target, dtype=torch.float32)
            except Exception as e:
                # Naya index try karo agar error aaye
                idx = random.randint(0, len(self.df)-1)
        
        # Extremely safe fallback (par ab iski zarurat nahi padni chahiye)
        return torch.zeros((3,128,128)), torch.zeros((3,128,128)), torch.tensor(0.0)

In [35]:
class SiameseDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.id2imgs = self.df.groupby('person_id')['img_path'].apply(list).to_dict()
        self.all_pids = list(self.id2imgs.keys())

    def __len__(self):
        return len(self.df)

    def load_img(self, path):
        # Agar CSV mein .png likha hai aur folder mein .webp hai, toh hum extension badal denge
        path = str(path).strip()
        if path.endswith('.png'):
            path = path.replace('.png', '.webp')
        
        full_path = os.path.join(self.img_dir, path)
        
        if not os.path.exists(full_path):
            # Agar abhi bhi nahi mili, toh loop retry karega
            raise FileNotFoundError
            
        return Image.open(full_path).convert('RGB')

    def __getitem__(self, idx):
        for _ in range(15): # Zyada retries
            try:
                row = self.df.iloc[idx]
                pid = row['person_id']
                target = random.randint(0, 1)
                
                img1 = self.load_img(row['img_path'])
                
                if target == 1:
                    img2_path = random.choice(self.id2imgs[pid])
                else:
                    other_pid = random.choice([p for p in self.all_pids if p != pid])
                    img2_path = random.choice(self.id2imgs[other_pid])
                
                img2 = self.load_img(img2_path)
                
                if self.transform:
                    img1, img2 = self.transform(img1), self.transform(img2)
                    
                return img1, img2, torch.tensor(target, dtype=torch.float32)
            except:
                idx = random.randint(0, len(self.df)-1)
        
        return torch.zeros((3,128,128)), torch.zeros((3,128,128)), torch.tensor(0.0)

In [39]:
# ============================================================
# CELL 4 — TRANSFORMS & DATALOADERS
# ============================================================
from torch.utils.data import DataLoader
from torchvision import transforms

# 1. Transforms: Image ko model ke layak banana
# Training ke liye augmentation (rotation/flip) taaki model robust bane
train_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# Validation ke liye sirf basic resize
val_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# 2. Dataset Objects (Jo humne Cell 3 mein define kiya tha)
# Ensure karna train_df aur val_df define ho chuke hain
train_ds = SiameseDataset(train_df, IMG_DIR, train_tf)
val_ds   = SiameseDataset(val_df,   IMG_DIR, val_tf)

# 3. DataLoaders: Batching aur Parallel Processing
train_loader = DataLoader(
    train_ds, 
    batch_size=32, 
    shuffle=True, 
    num_workers=2, 
    pin_memory=True
)

val_loader = DataLoader(
    val_ds, 
    batch_size=32, 
    shuffle=False, 
    num_workers=2, 
    pin_memory=True
)

print(f"✅ Dataloaders Ready!")
print(f"Total Training Batches: {len(train_loader)}")
print(f"Total Validation Batches: {len(val_loader)}")

# --- DEBUG CHECK ---
# Check karte hain ki pehla batch sahi aa raha hai ya nahi
img1, img2, label = next(iter(train_loader))
print(f"Batch Shape: {img1.shape}")
print(f"Max Pixel Value: {img1.max():.4f} (Should NOT be 0.0)")

✅ Dataloaders Ready!
Total Training Batches: 42
Total Validation Batches: 11
Batch Shape: torch.Size([32, 3, 128, 128])
Max Pixel Value: 1.0000 (Should NOT be 0.0)


In [43]:
import timm
import torch.nn as nn
import torch.optim as optim

# 1. Model Architecture (Fixed with 2112 dimension)
class EarSiameseModel(nn.Module):
    def __init__(self, embed_dim=256):
        super(EarSiameseModel, self).__init__()
        
        # Teeno backbones
        self.convnext = timm.create_model('convnext_atto', pretrained=True, num_classes=0)
        self.effnet   = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.resnet   = timm.create_model('resnet18', pretrained=True, num_classes=0)

        # Total features: 40 (Atto) + 1280 (EffNet) + 512 (ResNet18) = 1832
        # Par aapke model se 2112 aa rahe hain, toh hum wahi use karenge
        combined_feat_dim = 2112 

        self.projector = nn.Sequential(
            nn.Linear(combined_feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, embed_dim)
        )

    def forward_once(self, x):
        f1 = self.convnext(x)
        f2 = self.effnet(x)
        f3 = self.resnet(x)
        combined = torch.cat([f1, f2, f3], dim=1)
        embeddings = self.projector(combined)
        return nn.functional.normalize(embeddings, p=2, dim=1)

    def forward(self, img1, img2):
        return self.forward_once(img1), self.forward_once(img2)

# 2. Model Init
model = EarSiameseModel(embed_dim=256).to(DEVICE)

# 3. Contrastive Loss (Original Simple Version)
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, e1, e2, label):
        dist = torch.nn.functional.pairwise_distance(e1, e2, keepdim=True)
        loss = torch.mean(label * torch.pow(dist, 2) +
                          (1 - label) * torch.pow(torch.clamp(self.margin - dist, min=0.0), 2))
        return loss, dist

criterion = ContrastiveLoss(margin=1.0)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print("✅ Pehle wala model reset ho gaya! 2112 features fixed.")

✅ Pehle wala model reset ho gaya! 2112 features fixed.


In [44]:
# ============================================================
# STEP 6 — TRAINING LOOP
# ============================================================
from tqdm import tqdm

def accuracy(dists, labels, threshold=0.5):
    preds = (dists < threshold).float()
    return (preds == labels).float().mean()

EPOCHS = 15
best_acc = 0.0

print(f"Starting Training on {DEVICE}...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, train_acc = 0.0, 0.0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]')
    for img1, img2, labels in pbar:
        img1, img2, labels = img1.to(DEVICE), img2.to(DEVICE), labels.to(DEVICE).view(-1, 1)
        
        optimizer.zero_grad()
        e1, e2 = model(img1, img2)
        loss, dists = criterion(e1, e2, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_acc += accuracy(dists.detach(), labels)
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    # Validation
    model.eval()
    val_acc = 0.0
    with torch.no_grad():
        for img1, img2, labels in val_loader:
            img1, img2, labels = img1.to(DEVICE), img2.to(DEVICE), labels.to(DEVICE).view(-1, 1)
            e1, e2 = model(img1, img2)
            _, dists = criterion(e1, e2, labels)
            val_acc += accuracy(dists, labels)

    avg_train_loss = train_loss / len(train_loader)
    avg_train_acc = train_acc / len(train_loader)
    avg_val_acc = val_acc / len(val_loader)

    print(f"\nSummary Epoch {epoch}:")
    print(f"Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.2%} | Val Acc: {avg_val_acc:.2%}")

    if avg_val_acc > best_acc:
        best_acc = avg_val_acc
        torch.save(model.state_dict(), 'best_ear_model.pth')
        print(f"✨ Model Saved! Acc: {best_acc:.2%}")
    
    scheduler.step()
    print("-" * 40)

Starting Training on cuda...


Epoch 1/15 [Train]: 100%|██████████| 42/42 [00:12<00:00,  3.45it/s, loss=0.2838]



Summary Epoch 1:
Loss: 0.4675 | Train Acc: 49.98% | Val Acc: 55.97%
✨ Model Saved! Acc: 55.97%
----------------------------------------


Epoch 2/15 [Train]: 100%|██████████| 42/42 [00:12<00:00,  3.40it/s, loss=0.2308]



Summary Epoch 2:
Loss: 0.2656 | Train Acc: 50.92% | Val Acc: 47.73%
----------------------------------------


...


Summary Epoch 12:
Loss: 0.2462 | Train Acc: 53.89% | Val Acc: 43.41%
----------------------------------------


Epoch 13/15 [Train]: 100%|██████████| 42/42 [00:12<00:00,  3.37it/s, loss=0.2468]



Summary Epoch 13:
Loss: 0.2493 | Train Acc: 53.50% | Val Acc: 50.63%
----------------------------------------


Epoch 14/15 [Train]: 100%|██████████| 42/42 [00:12<00:00,  3.35it/s, loss=0.2592]


KeyboardInterrupt: 

In [36]:
# Ab check kar, pixel value 0 nahi honi chahiye
img1, img2, label = next(iter(train_loader))
print(f"Max pixel value: {img1.max()}") 
# Agar ye 0 se bada hai (like 1.0 or 0.8), toh BINGO!

Max pixel value: 0.0


In [32]:
import os
from PIL import Image
test_path = os.path.join(IMG_DIR, "0/0001.png") # CSV ki pehli entry ka path yahan likh
if os.path.exists(test_path):
    print("Mil gayi!")
    display(Image.open(test_path))
else:
    print(f"Nahi mili! Path check kar: {test_path}")

Nahi mili! Path check kar: /kaggle/input/datasets/madhvii0911/uercccc/UERC26_Oriented/data/sequestered_anonymized/0/0001.png


In [33]:
import os

# Kaggle datasets usually follow this structure
BASE = '/kaggle/input/uercccc/UERC26_Oriented/data/sequestered_anonymized'

if os.path.exists(BASE):
    print("Base path mil gaya!")
    # Check karein ki iske andar kya folders hain
    print("Folders inside BASE:", os.listdir(BASE)[:5]) 
else:
    print("Abhi bhi nahi mila. Ek kaam kar, 'Data' tab mein jaake path copy kar.")
    # Agar abhi bhi nahi mil raha, toh ye command chala poora structure dekhne ke liye:
    # !find /kaggle/input -maxdepth 4 -type d

Abhi bhi nahi mila. Ek kaam kar, 'Data' tab mein jaake path copy kar.


In [34]:
import os

# 1. Jo path tune copy kiya hai wahi yahan rakho
BASE = '/kaggle/input/datasets/madhvii0911/uercccc/UERC26_Oriented/data/sequestered_anonymized'

# 2. Check karte hain ki folder exist karta hai ya nahi
if os.path.exists(BASE):
    print("✅ BASE Path mil gaya!")
    
    # Check 1: Kya image seedha yahan hai? (0/0001.png)
    test1 = os.path.join(BASE, "0/0001.png")
    
    # Check 2: Kya ek aur extra folder hai beech mein? (UERC26_Oriented/0/0001.png)
    # Kabhi kabhi nesting double ho jati hai
    
    if os.path.exists(test1):
        print("🚀 BINGO! Image mil gayi: ", test1)
        IMG_DIR = BASE
    else:
        print("❌ BASE toh mila, par uske andar '0/0001.png' nahi hai.")
        print("Folder ke andar ye cheezein hain:", os.listdir(BASE))
        
        # Ek level aur andar check karte hain
        sub_folders = [f for f in os.listdir(BASE) if os.path.isdir(os.path.join(BASE, f))]
        if sub_folders:
            new_attempt = os.path.join(BASE, sub_folders[0], "0/0001.png")
            if os.path.exists(new_attempt):
                print(f"✅ Found it here instead: {os.path.join(BASE, sub_folders[0])}")
                IMG_DIR = os.path.join(BASE, sub_folders[0])
else:
    print("❌ Path abhi bhi galat hai. Username wala path dobara check kar.")

✅ BASE Path mil gaya!
❌ BASE toh mila, par uske andar '0/0001.png' nahi hai.
Folder ke andar ye cheezein hain: ['image_list.csv', 'dataset_split.csv', '0']


In [24]:
# ============================================================
# CELL 4 — TRANSFORMS & DATALOADERS
# ============================================================

# 1. Transforms (Image Preprocessing)
train_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5), # Flip taaki model left/right ear confuse na ho
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

val_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# 2. Dataset Objects
# Ensure karna ki train_df aur val_df pehle split ho chuke hain
train_ds = SiameseDataset(train_df, IMG_DIR, train_tf)
val_ds   = SiameseDataset(val_df,   IMG_DIR, val_tf)

# 3. DataLoaders
# Batch size 32 safe hai, num_workers=2 Kaggle ke liye best hai
train_loader = DataLoader(
    train_ds, 
    batch_size=32, 
    shuffle=True, 
    num_workers=2, 
    pin_memory=True
)

val_loader = DataLoader(
    val_ds, 
    batch_size=32, 
    shuffle=False, 
    num_workers=2, 
    pin_memory=True
)

print(f"Dataloaders Ready!")
print(f"Total Training Batches: {len(train_loader)}")
print(f"Total Validation Batches: {len(val_loader)}")

Dataloaders Ready!
Total Training Batches: 42
Total Validation Batches: 11


In [28]:
# ============================================================
# CELL 5 — MODEL ARCHITECTURE (Triple Backbone Siamese)
# ============================================================
import timm
import torch.nn as nn

class TripleBackboneSiamese(nn.Module):
    def __init__(self, embed_dim=256, pretrained=True):
        super(TripleBackboneSiamese, self).__init__()
        
        # 1. Teeno backbones (Feature Extractors)
        # num_classes=0 ka matlab hai hume sirf features chahiye, classification nahi
        self.convnext = timm.create_model('convnext_atto', pretrained=pretrained, num_classes=0)
        self.effnet   = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        self.resnet   = timm.create_model('resnet18', pretrained=pretrained, num_classes=0)

        # Teeno models ke output dimensions ko calculate karke combine kar rahe hain
        # Convnext (40) + EffNet (1280) + ResNet (512) = 1832
        combined_feat_dim = 2112

        # 2. Projector Head: Jo features ko final embedding mein convert karega
        self.projector = nn.Sequential(
            nn.Linear(combined_feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, embed_dim)
        )

    def encode(self, x):
        # Image se features nikalne ka flow
        f1 = self.convnext(x)
        f2 = self.effnet(x)
        f3 = self.resnet(x)
        
        # Teeno ko ek saath jodna
        combined = torch.cat([f1, f2, f3], dim=1)
        
        # Normalize karna zaruri hai distance calculation ke liye
        embeddings = self.projector(combined)
        return nn.functional.normalize(embeddings, p=2, dim=1)

    def forward(self, img1, img2):
        # Dono images ke liye same weights use honge (Siamese logic)
        return self.encode(img1), self.encode(img2)

# Model initialize karke GPU par bhejna
model = TripleBackboneSiamese(embed_dim=256).to(DEVICE)
print("Model Initialized and sent to DEVICE!")

Model Initialized and sent to DEVICE!


In [45]:
import timm
import torch.nn as nn
import torch.optim as optim

# 1. Wahi Model Architecture jo pehle kaam kar raha tha
class EarSiameseModel(nn.Module):
    def __init__(self, embed_dim=256):
        super(EarSiameseModel, self).__init__()
        
        # Teeno backbones load ho rahe hain
        self.convnext = timm.create_model('convnext_atto', pretrained=True, num_classes=0)
        self.effnet   = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.resnet   = timm.create_model('resnet18', pretrained=True, num_classes=0)

        # Dimension fixed as per your error (32x2112)
        combined_feat_dim = 2112 

        self.projector = nn.Sequential(
            nn.Linear(combined_feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, embed_dim)
        )

    def forward_once(self, x):
        f1 = self.convnext(x)
        f2 = self.effnet(x)
        f3 = self.resnet(x)
        
        combined = torch.cat([f1, f2, f3], dim=1)
        embeddings = self.projector(combined)
        
        # Normalize embeddings (L2 Norm) - Zaroori hai Siamese ke liye
        return nn.functional.normalize(embeddings, p=2, dim=1)

    def forward(self, img1, img2):
        return self.forward_once(img1), self.forward_once(img2)

# 2. Model Initialization
model = EarSiameseModel(embed_dim=256).to(DEVICE)

# 3. Original Contrastive Loss (Simple and Working)
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, e1, e2, label):
        dist = torch.nn.functional.pairwise_distance(e1, e2, keepdim=True)
        # Loss formula: Same image (label 1) -> dist kam karo, Different (label 0) -> dist badhao
        loss = torch.mean(label * torch.pow(dist, 2) +
                          (1 - label) * torch.pow(torch.clamp(self.margin - dist, min=0.0), 2))
        return loss, dist

criterion = ContrastiveLoss(margin=1.0)

# 4. Optimizer & Scheduler
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print("✅ Purana logic reset! Ab training start kar lo.")

✅ Purana logic reset! Ab training start kar lo.


In [30]:
# Check if data is actually coming through
img1, img2, label = next(iter(train_loader))
print(f"Image 1 Shape: {img1.shape}") # Should be [32, 3, 128, 128]
print(f"Labels: {label}")             # Should show 0s and 1s
print(f"Max pixel value: {img1.max()}") # Should not be 0.0

Image 1 Shape: torch.Size([32, 3, 128, 128])
Labels: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.])
Max pixel value: 0.0


In [46]:
# ============================================================
# STEP 6 — TRAINING LOOP
# ============================================================
import torch.nn.functional as F
from tqdm import tqdm

def accuracy(dists, labels, threshold=0.5):
    # Agar distance threshold se kam hai, toh model same person keh raha hai (label 1)
    preds = (dists < threshold).float()
    return (preds == labels).float().mean()

EPOCHS = 15
best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, train_acc = 0.0, 0.0
    
    # Training Loop
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]')
    for img1, img2, labels in pbar:
        img1, img2, labels = img1.to(DEVICE), img2.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
        
        optimizer.zero_grad()
        e1, e2 = model(img1, img2)
        loss, dists = criterion(e1, e2, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_acc += accuracy(dists.detach(), labels)
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    # Validation Loop
    model.eval()
    val_acc = 0.0
    with torch.no_grad():
        for img1, img2, labels in val_loader:
            img1, img2, labels = img1.to(DEVICE), img2.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
            e1, e2 = model(img1, img2)
            _, dists = criterion(e1, e2, labels)
            val_acc += accuracy(dists, labels)

    avg_train_loss = train_loss / len(train_loader)
    avg_train_acc = train_acc / len(train_loader)
    avg_val_acc = val_acc / len(val_loader)

    print(f"\nSummary Epoch {epoch}:")
    print(f"Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.2%} | Val Acc: {avg_val_acc:.2%}")

    # Best model save karein
    if avg_val_acc > best_acc:
        best_acc = avg_val_acc
        torch.save(model.state_dict(), 'best_ear_model.pth')
        print(f"✨ New Best Model Saved! Accuracy: {best_acc:.2%}")
    
    scheduler.step()
    print("-" * 40)

Epoch 1/15 [Train]: 100%|██████████| 42/42 [00:12<00:00,  3.35it/s, loss=0.2307]



Summary Epoch 1:
Loss: 0.4449 | Train Acc: 51.46% | Val Acc: 57.61%
✨ New Best Model Saved! Accuracy: 57.61%
----------------------------------------


Epoch 2/15 [Train]: 100%|██████████| 42/42 [00:12<00:00,  3.27it/s, loss=0.2912]



Summary Epoch 2:
Loss: 0.2625 | Train Acc: 50.64% | Val Acc: 49.72%
----------------------------------------


Epoch 3/15 [Train]: 100%|██████████| 42/42 [00:12<00:00,  3.26it/s, loss=0.2196]


...


Summary Epoch 13:
Loss: 0.2484 | Train Acc: 52.12% | Val Acc: 48.52%
----------------------------------------


Epoch 14/15 [Train]: 100%|██████████| 42/42 [00:12<00:00,  3.34it/s, loss=0.2396]



Summary Epoch 14:
Loss: 0.2485 | Train Acc: 53.32% | Val Acc: 47.67%
----------------------------------------


Epoch 15/15 [Train]: 100%|██████████| 42/42 [00:12<00:00,  3.36it/s, loss=0.2683]



Summary Epoch 15:
Loss: 0.2475 | Train Acc: 53.34% | Val Acc: 52.10%
----------------------------------------


In [47]:
torch.save(model.state_dict(), 'ear_siamese_final.pth')
print("✅ Model weights saved as ear_siamese_final.pth")

✅ Model weights saved as ear_siamese_final.pth


In [51]:
import torch
import os

# 1. Kaggle ka official output directory specify karo
SAVE_PATH = '/kaggle/working/best_ear_model.pth'

# 2. Model save karo
try:
    torch.save(model.state_dict(), SAVE_PATH)
    
    # 3. Double check ki file create hui ya nahi
    if os.path.exists(SAVE_PATH):
        file_size = os.path.getsize(SAVE_PATH) / (1024 * 1024) # MB mein size
        print(f"✅ SUCCESS! Model save ho gaya hai.")
        print(f"📍 Path: {SAVE_PATH}")
        print(f"📦 File Size: {file_size:.2f} MB")
        print("\nAb Right Sidebar mein 'Output' section refresh karo, wahan file mil jayegi.")
    else:
        print("❌ File save nahi ho paayi. Shayad model define nahi hai.")
except Exception as e:
    print(f"❌ Error aaya save karte waqt: {e}")

✅ SUCCESS! Model save ho gaya hai.
📍 Path: /kaggle/working/best_ear_model.pth
📦 File Size: 75.85 MB

Ab Right Sidebar mein 'Output' section refresh karo, wahan file mil jayegi.


In [52]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model.eval() # Evaluation mode
all_dists = []
all_labels = []
val_losses = []

print("📊 Calculating Model Performance...")

with torch.no_grad():
    for img1, img2, labels in val_loader:
        img1, img2, labels = img1.to(DEVICE), img2.to(DEVICE), labels.to(DEVICE).view(-1, 1)
        
        # Forward pass
        e1, e2 = model(img1, img2)
        loss, dists = criterion(e1, e2, labels)
        
        val_losses.append(loss.item())
        all_dists.extend(dists.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Thresholding (Dist < 0.5 means Same Person)
threshold = 0.5
preds = [1 if d < threshold else 0 for d in all_dists]

# Metrics Calculation
acc = accuracy_score(all_labels, preds)
avg_loss = np.mean(val_losses)
precision = precision_score(all_labels, preds)
recall = recall_score(all_labels, preds)
f1 = f1_score(all_labels, preds)

print("-" * 30)
print(f"📈 FINAL VALIDATION REPORT")
print("-" * 30)
print(f"✅ Accuracy  : {acc:.2%}")
print(f"📉 Avg Loss  : {avg_loss:.4f}")
print(f"🎯 Precision : {precision:.4f}")
print(f"📢 Recall    : {recall:.4f}")
print(f"⚖️  F1-Score  : {f1:.4f}")
print("-" * 30)

# Error Format (False Positives & Negatives)
tp = sum((np.array(preds) == 1) & (np.array(all_labels).flatten() == 1))
fp = sum((np.array(preds) == 1) & (np.array(all_labels).flatten() == 0))
fn = sum((np.array(preds) == 0) & (np.array(all_labels).flatten() == 1))
tn = sum((np.array(preds) == 0) & (np.array(all_labels).flatten() == 0))

print(f"❌ Errors (Galtiyan):")
print(f"   - False Positives (Galat Pehchaan): {fp}")
print(f"   - False Negatives (Pehchan nahi paya): {fn}")

📊 Calculating Model Performance...
------------------------------
📈 FINAL VALIDATION REPORT
------------------------------
✅ Accuracy  : 48.53%
📉 Avg Loss  : 0.2584
🎯 Precision : 0.4853
📢 Recall    : 1.0000
⚖️  F1-Score  : 0.6535
------------------------------
❌ Errors (Galtiyan):
   - False Positives (Galat Pehchaan): 175
   - False Negatives (Pehchan nahi paya): 0


In [49]:
model.eval() # Model ko evaluation mode mein daalo
total_dist = []
total_labels = []

print("Running Evaluation...")
with torch.no_grad():
    for img1, img2, labels in val_loader:
        img1, img2, labels = img1.to(DEVICE), img2.to(DEVICE), labels.to(DEVICE)
        
        e1, e2 = model(img1, img2)
        dist = torch.nn.functional.pairwise_distance(e1, e2)
        
        total_dist.extend(dist.cpu().numpy())
        total_labels.extend(labels.cpu().numpy())

print("Evaluation Done!")

# Check first 5 predictions
for i in range(5):
    prediction = "Same" if total_dist[i] < 0.5 else "Different"
    actual = "Same" if total_labels[i] == 1 else "Different"
    print(f"Pair {i+1}: Prediction={prediction}, Actual={actual} (Dist: {total_dist[i]:.4f})")

Running Evaluation...
Evaluation Done!
Pair 1: Prediction=Same, Actual=Same (Dist: 0.4049)
Pair 2: Prediction=Same, Actual=Same (Dist: 0.4390)
Pair 3: Prediction=Same, Actual=Same (Dist: 0.3682)
Pair 4: Prediction=Same, Actual=Same (Dist: 0.4056)
Pair 5: Prediction=Same, Actual=Same (Dist: 0.0000)
